# CS462 Lab 7 — Notebook 2: Spatio-Temporal Analysis with Apache Sedona

This notebook is **Part 4** of the lab. It picks up the feature frame you wrote at the
end of Notebook 1 (`photo_features.parquet`) and treats the extracted `lat`/`lon`/
`capture_time` columns as a spatio-temporal dataset, using **Apache Sedona**.

> **Prerequisite:** run Notebook 1 end-to-end first so `photo_features.parquet` exists in
> the work folder. This notebook reads it; it never touches image bytes.

Sedona adds a spatial **User-Defined Type** (`GeometryUDT`) plus `ST_*` SQL functions on
top of Spark. You'll build point geometries, join points to neighborhood polygons with a
spatial predicate, add a temporal filter, run a nearest-neighbor query, and finish with a
capstone that combines a *visual* feature (brightness) with a *spatial* grouping.

## Setup — a Sedona-enabled SparkSession (given)

The Sedona JARs are already baked into the image, so there's no `spark.jars.packages`
download here. `SedonaContext.create` registers the `ST_*` functions and the spatial type.

In [ ]:
from sedona.spark import SedonaContext

BUCKET = "s3a://torstengrabs-bc/lab7"

config = (
    SedonaContext.builder()
    .appName("lab7-sedona")
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
    .config("spark.kryo.registrator",
            "org.apache.sedona.core.serde.SedonaKryoRegistrator")
    .config("spark.hadoop.fs.s3a.aws.credentials.provider",
            "com.amazonaws.auth.profile.ProfileCredentialsProvider")
    .config("spark.hadoop.fs.s3a.requester.pays.enabled", "true")
    .config("spark.hadoop.fs.s3a.endpoint", "s3.us-east-1.amazonaws.com")
    .getOrCreate()
)
sedona = SedonaContext.create(config)
print("Sedona ready on Spark", sedona.version)

### 4.1  Load the feature frame and build point geometries

Load the Parquet written by Notebook 1, drop rows that had no GPS, then build a `geom`
column. **Fill in the `ST_Point` call** (it takes longitude first, then latitude — the
x, y convention). This mirrors the `ST_Point(CAST(lon ...), CAST(lat ...))` pattern from
lecture.

In [ ]:
features = sedona.read.parquet("file:///home/jovyan/work/photo_features.parquet")
features = features.filter("lat IS NOT NULL AND lon IS NOT NULL")
features.createOrReplaceTempView("photos_raw")
print("geotagged photos:", features.count())

In [ ]:
points = sedona.sql("""
    SELECT path, brightness, capture_time,
           -- TODO: build a point geometry from lon, lat (x = lon, y = lat)
           <ST_Point(...)> AS geom
    FROM photos_raw
""")
points.createOrReplaceTempView("points")
points.show(5, truncate=False)

### 4.2  Load the neighborhood polygons (given)

The neighborhoods come as WKT strings in a CSV. `ST_GeomFromText` parses each into a
polygon geometry.

In [ ]:
zones = (
    sedona.read.option("header", True).csv(f"{BUCKET}/neighborhoods.csv")
    .selectExpr("name AS zone_name", "ST_GeomFromText(wkt) AS boundary")
)
zones.createOrReplaceTempView("zones")
print("zones:", zones.count())
zones.select("zone_name").show(truncate=False)

### 4.3  Exercise — daytime photos per neighborhood (12 pts)

Write a spatial **and** temporal query: count how many photos fall **inside** each
neighborhood polygon **and** were taken during daylight working hours (capture hour
between 9 and 17 inclusive). Group by `zone_name`, order by count descending.

Use `ST_Contains(zone.boundary, point.geom)` for the spatial predicate and `hour(...)`
for the temporal one. Expected: **8 rows, 78 photos total.**

In [ ]:
daytime = sedona.sql("""
    -- TODO: count photos that are ST_Contains-ed by each zone AND whose
    --       capture hour is between 9 and 17, grouped by zone, ordered desc.
""")
daytime.show()

### 4.4  Exercise — five photos nearest the Space Needle (8 pts)

Load the landmarks CSV, pull the Space Needle's coordinates, and find the 5 photos taken
closest to it. Use `ST_DistanceSphere(geom, ST_Point(lon, lat))` (returns meters) and
`ORDER BY ... LIMIT 5`. Expected: **5 rows** (all happen to be in Queen Anne).

In [ ]:
landmarks = sedona.read.option("header", True).csv(f"{BUCKET}/landmarks.csv")
needle = landmarks.filter("name = 'Space Needle'").collect()[0]
n_lat, n_lon = float(needle["lat"]), float(needle["lon"])
print("Space Needle:", n_lat, n_lon)

In [ ]:
nearest = sedona.sql(f"""
    -- TODO: select path, capture_time, and the distance in meters from each
    --       point to the Space Needle (ST_DistanceSphere). Order ascending,
    --       limit 5. The query is an f-string, so {n_lon} and {n_lat} get
    --       replaced with the Space Needle's coordinates.
""")
nearest.show(truncate=False)

### 4.5  Plan inspection & spatial indexing — bonus (10 pts)

Call `.explain()` on your 4.3 query and look at how Sedona executed the join. A naive
spatial join would test every (zone, point) pair — O(zones × points). Sedona instead
rewrites an `ST_Contains` join into an indexed spatial join.

Run the cell, then in the markdown cell after it answer: (a) what join operator name does
Sedona use in the plan (look for a spatial / range / index join rather than a plain
`BroadcastNestedLoopJoin`)? (b) Why does a spatial index matter as the number of points
grows — what does it save you compared with the all-pairs approach (lecture slide 22)?

In [ ]:
daytime.explain()

In [ ]:
# Bonus answer:
# (a) Which join operator does Sedona use in the plan?
#
# (b) Why does the spatial index matter as the point count grows?

### 4.6  Capstone — average brightness per neighborhood (5 pts)

Bring the two halves of the lab together. Using the same spatial join as 4.3 (but **no**
time filter this time), compute, per neighborhood: the photo count and the **average
brightness** (the visual feature you extracted in Notebook 1), ordered by average
brightness descending. Expected: **8 rows, 231 photos total.**

In [ ]:
capstone = sedona.sql("""
    -- TODO: per zone, COUNT(*) and AVG(brightness) for all photos contained
    --       in the zone (no time filter). Order by avg brightness desc.
""")
capstone.show()

### 4.7  Spark UI screenshot (part of 4.5 / 4.6 credit)

Save a screenshot of the **Stages** tab for the capstone query as
`notebook2_screenshots/capstone_dag.png`.

---
## Part 5 — Reflection

Answer the three questions in the handout (Part 5) in a separate file `reflection.md`.
They ask you to connect what you did here to the lecture: media-vs-tabular processing and
avoiding blob shuffles (Notebook 1, §2.5); UDF vs UDT and the flavors of UDFs (Notebook
1, Part 3); and why a spatial join is hard without an extension like Sedona (Notebook 2,
§4.5).